# Exercice 09 - Nettoyage des donnees (Silver)

## Objectifs
- Comprendre la couche Silver du Data Lake
- Gerer les valeurs nulles
- Supprimer les doublons
- Standardiser les formats de donnees
- Valider la qualite des donnees

---

## 1. La couche Silver

```
+------------------+     +------------------+     +------------------+
|      BRONZE      |     |      SILVER      |     |       GOLD       |
+------------------+     +------------------+     +------------------+
|                  |     |                  |     |                  |
| Donnees brutes   | --> | Donnees propres  | --> | Donnees business |
| Non validees     |     | Validees         |     | Agregatees       |
| Avec doublons    |     | Sans doublons    |     | Optimisees       |
| Formats varies   |     | Formats standards|     | Pretes a l'usage |
|                  |     |                  |     |                  |
+------------------+     +------------------+     +------------------+

Nettoyages Silver :
- Suppression des doublons
- Gestion des valeurs nulles
- Standardisation des formats
- Validation des types
- Correction des erreurs evidentes
```

## 2. Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType, DateType
from datetime import datetime

# Creer la SparkSession
spark = SparkSession.builder \
    .appName("Nettoyage Silver") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

date_traitement = datetime.now().strftime("%Y-%m-%d")
print(f"Spark pret - Date : {date_traitement}")

## 3. Creer des donnees de test avec des problemes

In [ ]:
# Donnees avec des problemes courants
data_problemes = [
    (1, "Alice", "alice@email.com", "Paris", 25, 50000.0),
    (2, "Bob", "BOB@EMAIL.COM", "paris", 30, 60000.0),      # Email en majuscules, ville minuscule
    (3, "Charlie", None, "Lyon", 35, None),                  # Valeurs nulles
    (4, "Diana", "diana@email.com", "  Marseille  ", -5, 70000.0),  # Espaces, age negatif
    (1, "Alice", "alice@email.com", "Paris", 25, 50000.0),   # Doublon
    (5, "Eve", "eve@email", "Toulouse", 28, 55000.0),        # Email invalide
    (6, "Frank", "frank@email.com", "", 40, 80000.0),        # Ville vide
    (7, "Grace", "grace@email.com", "Nice", 150, 65000.0),   # Age impossible
]

colonnes = ["id", "nom", "email", "ville", "age", "salaire"]
df_bronze = spark.createDataFrame(data_problemes, colonnes)

print("Donnees brutes (Bronze) :")
df_bronze.show(truncate=False)

## 4. Supprimer les doublons

In [ ]:
# Compter avant
print(f"Lignes avant : {df_bronze.count()}")

# Supprimer les doublons exacts
df_sans_doublons = df_bronze.dropDuplicates()
print(f"Lignes apres dropDuplicates() : {df_sans_doublons.count()}")

# Supprimer les doublons sur certaines colonnes
df_sans_doublons_id = df_bronze.dropDuplicates(["id"])
print(f"Lignes apres dropDuplicates(['id']) : {df_sans_doublons_id.count()}")

In [ ]:
# Continuer avec les donnees sans doublons
df = df_sans_doublons
df.show()

## 5. Gerer les valeurs nulles

In [ ]:
# Compter les valeurs nulles par colonne
print("Valeurs nulles par colonne :")
for col in df.columns:
    nb_nulls = df.filter(F.col(col).isNull()).count()
    print(f"  {col}: {nb_nulls}")

In [ ]:
# Option 1 : Supprimer les lignes avec des nulls
df_drop_na = df.dropna()
print(f"Apres dropna() : {df_drop_na.count()} lignes")

In [ ]:
# Option 2 : Remplacer les nulls par des valeurs par defaut
df_fill = df.fillna({
    "email": "inconnu@example.com",
    "salaire": 0.0
})

df_fill.show()

In [ ]:
# Option 3 : Remplacer par la moyenne (pour les numeriques)
moyenne_salaire = df.agg(F.avg("salaire")).collect()[0][0]
print(f"Salaire moyen : {moyenne_salaire}")

df_fill_avg = df.fillna({"salaire": moyenne_salaire})
df_fill_avg.show()

## 6. Standardiser les formats

In [ ]:
# Appliquer plusieurs nettoyages
df_clean = df_fill \
    .withColumn("email", F.lower(F.col("email"))) \
    .withColumn("ville", F.trim(F.col("ville"))) \
    .withColumn("ville", F.initcap(F.col("ville"))) \
    .withColumn("nom", F.initcap(F.col("nom")))

print("Apres standardisation :")
df_clean.show(truncate=False)

In [ ]:
# Traiter les villes vides
df_clean = df_clean.withColumn(
    "ville",
    F.when(F.col("ville") == "", "Inconnue").otherwise(F.col("ville"))
)

df_clean.show(truncate=False)

## 7. Valider les donnees

In [ ]:
# Valider l'age (entre 0 et 120)
df_valid = df_clean.withColumn(
    "age_valide",
    F.when((F.col("age") >= 0) & (F.col("age") <= 120), True).otherwise(False)
)

print("Ages invalides :")
df_valid.filter(F.col("age_valide") == False).show()

In [ ]:
# Corriger ou filtrer les ages invalides
df_age_corrige = df_clean.withColumn(
    "age",
    F.when(F.col("age") < 0, None)
     .when(F.col("age") > 120, None)
     .otherwise(F.col("age"))
)

df_age_corrige.show()

In [ ]:
# Valider le format email avec regex
pattern_email = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

df_email_valide = df_age_corrige.withColumn(
    "email_valide",
    F.col("email").rlike(pattern_email)
)

print("Emails invalides :")
df_email_valide.filter(F.col("email_valide") == False).show(truncate=False)

## 8. Pipeline de nettoyage complet

In [ ]:
def nettoyer_donnees(df):
    """
    Pipeline de nettoyage complet.
    
    Args:
        df: DataFrame brut (Bronze)
    
    Returns:
        DataFrame nettoye (Silver)
    """
    # 1. Supprimer les doublons
    df = df.dropDuplicates(["id"])
    
    # 2. Gerer les nulls
    df = df.fillna({
        "email": "inconnu@example.com",
        "ville": "Inconnue",
        "salaire": 0.0
    })
    
    # 3. Standardiser les formats
    df = df.withColumn("email", F.lower(F.trim(F.col("email")))) \
           .withColumn("ville", F.initcap(F.trim(F.col("ville")))) \
           .withColumn("nom", F.initcap(F.trim(F.col("nom"))))
    
    # 4. Traiter les villes vides
    df = df.withColumn(
        "ville",
        F.when(F.col("ville") == "", "Inconnue").otherwise(F.col("ville"))
    )
    
    # 5. Valider et corriger l'age
    df = df.withColumn(
        "age",
        F.when((F.col("age") >= 0) & (F.col("age") <= 120), F.col("age"))
         .otherwise(None)
    )
    
    # 6. Ajouter les metadonnees
    df = df.withColumn("_cleaned_date", F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
    
    return df

In [ ]:
# Appliquer le pipeline
df_silver = nettoyer_donnees(df_bronze)

print("Donnees nettoyees (Silver) :")
df_silver.show(truncate=False)

## 9. Rapport de qualite

In [ ]:
def rapport_qualite(df_avant, df_apres):
    """
    Genere un rapport de qualite des donnees.
    """
    print("=" * 50)
    print("RAPPORT DE QUALITE")
    print("=" * 50)
    
    print(f"\nLignes avant  : {df_avant.count()}")
    print(f"Lignes apres  : {df_apres.count()}")
    print(f"Lignes retirees: {df_avant.count() - df_apres.count()}")
    
    print("\nValeurs nulles (apres nettoyage) :")
    for col in df_apres.columns:
        if not col.startswith("_"):
            nb_nulls = df_apres.filter(F.col(col).isNull()).count()
            pct = (nb_nulls / df_apres.count()) * 100 if df_apres.count() > 0 else 0
            print(f"  {col}: {nb_nulls} ({pct:.1f}%)")
    
    print("=" * 50)

rapport_qualite(df_bronze, df_silver)

## 10. Sauvegarder dans Silver

In [ ]:
# Sauvegarder les donnees nettoyees
chemin_silver = f"s3a://silver/users/{date_traitement}"
df_silver.write.mode("overwrite").parquet(chemin_silver)

print(f"Sauvegarde Silver : {chemin_silver}")

---

## Exercice

**Objectif** : Nettoyer les donnees Northwind customers

**Consigne** :
1. Lisez la table `customers` depuis PostgreSQL ou Bronze
2. Appliquez les nettoyages suivants :
   - Standardisez les noms de pays (majuscules)
   - Supprimez les espaces en debut/fin de texte
   - Gerez les valeurs nulles
3. Sauvegardez dans Silver

A vous de jouer :

In [ ]:
# TODO: Lire les customers depuis PostgreSQL
jdbc_url = "jdbc:postgresql://postgres:5432/app"
jdbc_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

df_customers = spark.read.jdbc(url=jdbc_url, table="customers", properties=jdbc_properties)
df_customers.show(5)

In [ ]:
# TODO: Appliquer les nettoyages

In [ ]:
# TODO: Sauvegarder dans Silver

---

## Resume

Dans ce notebook, vous avez appris :
- Le role de la couche **Silver** dans un Data Lake
- Comment **supprimer les doublons** avec dropDuplicates()
- Comment **gerer les valeurs nulles** avec dropna() et fillna()
- Comment **standardiser les formats** (trim, lower, initcap)
- Comment **valider les donnees** avec des regles metier
- Comment creer un **pipeline de nettoyage** reutilisable

### Prochaine etape
Dans le prochain notebook, nous apprendrons les transformations avancees pour la couche Silver.